In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from non_linear_string import NonLinearString

## Nonlinear string equation

### String formulation

The following notebook shows how to simulate the string equation with fixed edges. The conservative string equation is a particularization of the wave equation, where the so-called geometrically nonlinear string model from **Morse and Ingard** (1968) states that

$$ (\mu\partial_{t}^2-T_0\partial_x^2+EI\partial_{x}^4)u=\partial_x\left(\frac{\partial\phi}{\partial(\partial_x u)}\right)-(2\eta_0-2\eta_1\partial_x^2)\partial_tu+\delta(x-x_{in})f_{in} $$

where $\mu=\rho A$ denotes the lineic density of mass with $\rho$ being the density and $A$ the cross section. $T$ denotes the tension, $I$ the inertia momentum, $E$ the Young's module in Pa and dissipation coefficients $\eta_0$ and $\eta_1$.



In the following, we simulate a E2 string being plucked using port-Hamiltonian formulation. In the view of real-time application, it is useful to recall some known results on the eigenvalues of the linearized system. First, the imaginary part of the eigenvalues of the linearized system is studied, neglecting losses. This yields the following series of eigenfrequencies

$$ f_n\approx (n+1)\frac{c_0}{2l_0}\sqrt{1+\beta(n+1)^2}, \hspace{1cm} c_0=\sqrt{\frac{T_0}{\mu}}, \hspace{1cm} \beta=\frac{\pi^2EI}{T_0l_0^2}  $$

where $f_0$ is the fundamental frequency and $\beta$ is a measure of the inharmonicity. This parametrization is convenient as it exposes perceptually relevant information. Of course addition of losses is expected to modify the series, but for musical strings, dissipation is small enough to be neglected in this approximation.

For the simulation, the spatial grid is set to have 100 points and we set a timestep corresponding to a sampling rate of 44.1kHz.

In [ ]:
# Parameters from a guitar string (E2).
string = NonLinearString(L=0.65, R=5.97e-4, rho=7850, T=90.48, E=2.1e8, eta0=0.9, eta1=2e-4, num_spatial_points=100)

# Fundamental frequency calculation
print(f'Fundamental frequency: {string.f0:.1f}Hz')
print(f'Inharmonicity: {string.beta:.1e}')

# Temporal parameters
sampling_rate = 88.2e3
timestep = 1 / sampling_rate
duration = 2
t_axis = np.arange(0, duration, timestep)

# Print basic information about the grid.
print(f'The spatial grid is of width: {string.h:.2e}')
print(f'The time grid is of width: {timestep:.2e}.')

### Port-Hamiltonian formulation and simulation of a string

We will transform this 2nd-order in time PDE into a 1st-order in time PDE. For this purpose, we introduce the states 

$$\alpha:=\begin{bmatrix}\pi:=\mu \partial_tu \\ \epsilon:=\partial_xu \\ \epsilon_x:=\partial_x^2u \end{bmatrix}$$

where $\pi$ denotes the lineic momentum and $\epsilon$ the strain. This allow us to write the Hamiltonian of the system as 

$$H(t)=\int_0^L \left(\frac{1}{2\mu}\partial_t^2u+\frac{T}{2}\partial_x^2u+\frac{EI}{2}\partial_x^4u+\frac{EA-T_0}{8}\partial_x^4u\right)dx= \int_0^L \left(\frac{\alpha_0^2}{2\mu}+\frac{T}{2}\alpha_1^2+\frac{EI}{2}\alpha_2^2+\frac{EA-T_0}{8}\alpha_{1}^4\right) dx$$

This is the continuous definition, which in the simulation is approximated as 

$$H(t)\approx h\sum_{i=0}^{M-1} \left(\frac{\alpha_{0,i}^2}{2}+\frac{T}{2}\alpha_{1,i}^2+ \frac{EI}{2}\alpha_{2,i}^2+\frac{EA-T_0}{8}\alpha_{1,i}^4\right)=\frac{1}{2}\|\alpha_0 \|^2+ \frac{T}{2}\|\alpha_1\|^2+\frac{EI}{2}\|\alpha_2\|^2+\frac{EA-T_0}{8}\|\alpha_1^2 \|^2$$

In [ ]:
# Parameters.
mu, tension, E, I, A = string.mu, string.T, string.E, string.I, string.A
num_pi, num_strain = string.num_pi, string.num_strain
dx = string.h

# Define the Hamiltonian and its gradient.
def hamiltonian(x: np.ndarray, mu=mu, T=tension) -> np.ndarray:
    integrand = (mu * x[..., 0]**2) + (T * x[..., 1]**2) + (E * I / 2) * x[..., 2]**2 + ((E * A - T) / 8) * x[..., 1]**2
    return 0.5 * np.sum(integrand, axis=-1)

def hamiltonian_grad(x: np.ndarray) -> np.ndarray:
    e_pi = (1 / mu) * x[..., :num_pi]
    e_strain = x[..., num_pi:num_pi + num_strain] * (tension + ((E * A - tension) / 2) * x[..., num_pi:num_pi + num_strain]**2)
    e_strain_x = E * I * x[..., num_pi + num_strain:]
    return dx * np.concatenate((e_pi, e_strain, e_strain_x), axis=-1)

The port-Hamiltonian formulation of the conservative string equation can then simply be written as 

$$ \begin{bmatrix} \pi_t \\ \epsilon_t   \end{bmatrix} = \begin{bmatrix} 0 & \partial_x \\ \partial_x & 0 \end{bmatrix}\begin{bmatrix} \delta_{\pi}H \\ \delta_{\epsilon}H \end{bmatrix} $$

**Spatial disretization**. As fixed boundary conditions are considered ($u_0(t)=u_N(t)=0,\forall t\in\mathbb{R}^{+}$), the transverse displacement $u(x,t)$ is approximated by the displacements at the $N-1$ interior nodes of the grid

$$\mathbf{u}(t)=[u_1(t),\cdots,u_i(t),\cdots,u_{N-1}(t)]^T $$ 

 In order to build derivatives of the displacement inside the interconnection matrix, a finite difference matrix $D^{-}$ of first oder is used with Dirichlet boundary conditions.

$$ D^{-}=\frac{1}{h}\begin{bmatrix} 1 & 0 & 0  \\  -1 & \ddots & 0  \\ 0 & \ddots & 1  \\  0 & 0 & -1    \end{bmatrix}\in\mathbb{R}^{dim(\epsilon)\times dim(\pi)}$$

where $dim(\epsilon)=N$ and $dim(\pi)=N-1$. This rectangle matrix generates an approximation of the first spatial derivative of the displacement at the nodes of a "dual" grid, which is staggered with respect to the primal grid. As $\mathbf{u}$ is defined on the primal grid, the momentum $\mathbf{p}\equiv \mu u_t$ is also defined on the primal grid. The first spatial derivative of the displacement is obtained as $\epsilon=D^{-}\mathbf{u}$ and is thus defined on the dual grid.

Note: See [1] **Finite differences on staggered grids preserving the port-Hamiltonian structure with application to an acoustic duct** (https://www.sciencedirect.com/science/article/abs/pii/S0021999118304297) for a complete introduction to this dual grid concept.

Finally, the second spatial derivative of the displacement is obtained as $\partial_{xx}=D^{+}\epsilon$ with 

$$ D^{+}=-(D^{-})^T\in\mathbb{R}^{dim(\pi)\times dim(\epsilon)} $$

The discrete system dynamic then directly follows [1] as

$$ \begin{bmatrix} \pi_t \\ \epsilon_t \\ \epsilon_{x,t} \end{bmatrix}=\frac{1}{h}\left(\begin{bmatrix} -2\mu(\eta_0\mathbf{I}-\eta_1\mathbf{D}^2) & D^{+} & -D^2 \\ D^{-} & \mathbf{0}^{N\times N} & \mathbf{0}^{N\times N-1} \\ D^2 & \mathbf{0}^{N-1\times N} & \mathbf{0}^{N-1\times N-1} \end{bmatrix}\begin{bmatrix} \delta_{\pi}H \\ \delta_{\epsilon}H \\ \delta_{\epsilon_x}H \end{bmatrix}+\begin{bmatrix} \delta(x-x_e)f_{in} \\ \mathbf{0}^{N} \\ \mathbf{0}^{N-1} \end{bmatrix}\right)$$

We will impose Dirichlet boundary conditions ($w(t,x)=0$ on $(0,\infty)\times\{0,L\}$) and zero initial conditions ($w=0$ on $\{0\}\times(0,L)$). 

The string is excited by a pointwise external force $\mathcal{F}_e$, which can be modelled as

$$ \mathcal{F}_e(x,t)=\delta(x-x_e)f_e(t) $$

where $\delta(x-x_e)$ is the Dirac delta function at the excitation position $x_e$. Function $f_e(t)$ resembles a pluck of a string and is of the following form 

$$ f_e(t)=\begin{cases} \frac{1}{2}f_{amp}[1-cos\left(\frac{\pi t}{T_e} \right)], \hspace{1cm} t\in[0,T_e] \\ 0 \hspace{1cm} \text{otherwise} \end{cases} $$

In [ ]:
# Define an external control that will represent a pluck in the string. 
f_amp = 10
excitation_duration = 1e-3
def string_pluck_generator():
    # Choose a random index to apply the pluck.
    pluck_position = np.random.choice(np.arange(1, num_pi - 1))

    def string_pluck(t):
        force = f_amp * (1 - np.cos(np.pi * t / excitation_duration)) if t <= excitation_duration else 0
        input_effort = np.zeros((2 * string.num_pi + string.num_strain))
        input_effort[pluck_position] = force
        return input_effort
    
    return string_pluck

In [ ]:
# Stability conditions for the string simulation.
gamma = timestep**2 * tension + (4 * timestep * mu * string.eta1)
h_min = np.sqrt((gamma + np.sqrt(gamma**2 + 16 * mu * E * I * timestep**2)) / (2 * mu))
kappa = 0.9
N = 2 * ((kappa * string.L) / (2 * h_min))
print(f'Stability parameters: gamma={gamma:.2e}, h_min={h_min:.2e}, N={N:.2e}')

Simulate the trajectory using a RK4 solver and the spatial discretization mentioned above.

In [ ]:
# Initial conditions
num_trajectories = 1
x0 = np.zeros((2 * num_pi + num_strain))
# Generate the trajectories.
states = string.solve(
    num_trajectories=num_trajectories, duration=duration, sampling_rate=sampling_rate, x0=x0, control_generator=string_pluck_generator
)

### Simulation visualization

Once obtained the simulation for all the space points, we can see its representation with respect to time using a video.

**NOTE**: It may take a while to be created!

In [ ]:
from tools import create_video

x = np.linspace(0, string.L - string.L / string.N, string.N)

create_video(
    [states[0, ..., :num_pi]],
    ["Lineic momentum"],
    x_axis=x[1:-1],
    fps=int(0.001 / timestep)
)

We can also see how each of spatial grid nodes oscillates by plotting its trajectory.

In [ ]:
# Take the middle point inside the momentum results.
position = num_pi // 2
sampled_point_trajectory = states[0, ..., position]
plt.plot(t_axis, sampled_point_trajectory)
plt.title(f'Oscillation at the point in position {position}')
plt.xlabel('Time [s]')
plt.ylabel('Momentum')

The sampled trajectory can then be saved as an audio!

In [ ]:
from scipy.io.wavfile import write
# Normalize to audio range [-1, 1]
audio = sampled_point_trajectory / np.max(np.abs(sampled_point_trajectory))
filename = 'E2_string.wav'
write(filename, int(sampling_rate), audio.astype(np.float32))


The spectrogram can also be computed to check if we are synthesizing the desired frequency.

In [ ]:
from scipy.signal import spectrogram
from scipy.io.wavfile import read
import matplotlib.pyplot as plt

# Load the .wav file.
sampling_rate, audio_data = read(f'E2_string.wav')

# Compute the spectrogram.
frequencies, times, Sxx = spectrogram(audio_data, fs=sampling_rate, window='hann', nperseg=4096 * 2, noverlap=512, scaling='spectrum')
# Add a small epsilon.
epsilon = 1e-10
Sxx = Sxx + epsilon

# Step 3: Plot the spectrogram
plt.figure(figsize=(10, 6))
plt.pcolormesh(times, frequencies, 10 * np.log10(Sxx), shading='gouraud')
plt.title('Spectrogram')
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [sec]')
plt.colorbar(label='Power/Frequency (dB/Hz)')
plt.ylim(0, 6000)
plt.legend()
plt.savefig(f'{filename}.pdf')

### FURTHER WORK

i) Generate a dataset. 

ii) Design the learning framework of PHNNs for PDEs (in particular the string equation)

iii) Run experiments 